In [9]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import pickle

In [14]:
file = r"D:\Projects\MediGest\data\Preprocessed Medicine Dataset.csv"
data = pd.read_csv(file)
data = pd.DataFrame(data)
data.head()

,name,Chemical Class,Habit Forming,Therapeutic Class,Action Class,Substitutes,Side Effects,Uses
0,augmentin 625 duo tablet,NaN,No,ANTI INFECTIVES,NaN,"Penciclav 500 mg/125 mg Tablet, Moxikind-CV 62...","Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections
1,azithral 500 tablet,Macrolides,No,ANTI INFECTIVES,Macrolides,"Zithrocare 500mg Tablet, Azax 500 Tablet, Zady...","Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections
2,ascoril ls syrup,NaN,No,RESPIRATORY,NaN,"Solvin LS Syrup, Ambrodil-LX Syrup, Zerotuss X...","Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus
3,allegra 120mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex Tablet, Etofex 120mg Tablet, Nexofex 120...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...
4,avil 25 tablet,Pyridines Derivatives,No,RESPIRATORY,H1 Antihistaminics (First Generation),Eralet 25mg Tablet,"Sleepiness, Dryness in mouth",Treatment of Allergic conditions


In [15]:
data.isnull().sum()

name                      0
Chemical Class       110427
Habit Forming             0
Therapeutic Class        69
Action Class         110182
Substitutes            9597
Side Effects              0
Uses                      0
dtype: int64

In [16]:
data['Chemical Class'] = data['Chemical Class'].fillna('unknown')
data['Action Class'] = data['Action Class'].fillna('unknown')
data['Therapeutic Class'] = data['Therapeutic Class'].fillna('unknown')
data['Substitutes'] = data['Substitutes'].fillna('unknown')

Confidence score is calculated because there is some unknown data. If there is more unknown data, the score is high and vice-versa. The higher the score, the lesser the reliability. 

In [17]:
def compute_confidence(row):
    score = 0
    if row['Chemical Class'] != 'unknown':
        score += 1
    if row['Action Class'] != 'unknown':
        score += 1
    if row['Therapeutic Class'] != 'unknown':
        score += 1
    if row['Substitutes'] != 'unknown':
        score += 1

    return score

data['Confidence Score'] = data.apply(compute_confidence, axis = 1)
data.head()

,name,Chemical Class,Habit Forming,Therapeutic Class,Action Class,Substitutes,Side Effects,Uses,Confidence Score
0,augmentin 625 duo tablet,unknown,No,ANTI INFECTIVES,unknown,"Penciclav 500 mg/125 mg Tablet, Moxikind-CV 62...","Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections,2
1,azithral 500 tablet,Macrolides,No,ANTI INFECTIVES,Macrolides,"Zithrocare 500mg Tablet, Azax 500 Tablet, Zady...","Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections,4
2,ascoril ls syrup,unknown,No,RESPIRATORY,unknown,"Solvin LS Syrup, Ambrodil-LX Syrup, Zerotuss X...","Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus,2
3,allegra 120mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex Tablet, Etofex 120mg Tablet, Nexofex 120...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,4
4,avil 25 tablet,Pyridines Derivatives,No,RESPIRATORY,H1 Antihistaminics (First Generation),Eralet 25mg Tablet,"Sleepiness, Dryness in mouth",Treatment of Allergic conditions,4


Creating documents

In [18]:
def create_document(row):
    document = f"Name: {row['name']}\n"

    if row['Chemical Class'] != 'unknown':
        document += f"Chemical Class: {row['Chemical Class']}\n"
    if row['Action Class'] != 'unknown':
        document += f"Action Class: {row['Action Class']}\n"
    if row['Therapeutic Class'] != 'unknown':
        document += f"Therapeutic Class: {row['Therapeutic Class']}\n"
    if row['Substitutes'] != 'unknown':
        document += f"Substitutes: {row['Substitutes']}"

    document += f"""
        Habit Forming: {row['Habit Forming']}\n
        Side Effects: {row['Side Effects']}\n
        Uses: {row['Uses']}\n
        Confidence Score: {row['Confidence Score']}\n
        """

    return document

data['Docs'] = data.apply(create_document, axis = 1)
data.head()
        

,name,Chemical Class,Habit Forming,Therapeutic Class,Action Class,Substitutes,Side Effects,Uses,Confidence Score,Docs
0,augmentin 625 duo tablet,unknown,No,ANTI INFECTIVES,unknown,"Penciclav 500 mg/125 mg Tablet, Moxikind-CV 62...","Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections,2,Name: augmentin 625 duo tablet\nTherapeutic Cl...
1,azithral 500 tablet,Macrolides,No,ANTI INFECTIVES,Macrolides,"Zithrocare 500mg Tablet, Azax 500 Tablet, Zady...","Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections,4,Name: azithral 500 tablet\nChemical Class: Mac...
2,ascoril ls syrup,unknown,No,RESPIRATORY,unknown,"Solvin LS Syrup, Ambrodil-LX Syrup, Zerotuss X...","Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus,2,Name: ascoril ls syrup\nTherapeutic Class: RES...
3,allegra 120mg tablet,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation),"Lcfex Tablet, Etofex 120mg Tablet, Nexofex 120...","Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,4,Name: allegra 120mg tablet\nChemical Class: Di...
4,avil 25 tablet,Pyridines Derivatives,No,RESPIRATORY,H1 Antihistaminics (First Generation),Eralet 25mg Tablet,"Sleepiness, Dryness in mouth",Treatment of Allergic conditions,4,Name: avil 25 tablet\nChemical Class: Pyridine...


In [9]:
documents = data['Docs'].tolist()

with open("documents.pkl", 'wb') as f:
    pickle.dump(documents, f)

In [21]:
records = []

for _, row in data.iterrows():

    record = {

        "name": row["name"],

        "uses": row["Uses"],

        "side_effects": row["Side Effects"],

        "therapeutic_class": row["Therapeutic Class"],

        "action_class": row.get("Action Class", ""),

        "chemical_class": row.get("Chemical Class", ""),

        "confidence_score": row["Confidence Score"],

        "document": row["Docs"]
    }

    records.append(record)

In [ ]:
with open(r"data/records.pkl", "wb") as f:
    pickle.dump(records, f)

Model and Indexing

In [7]:
model = SentenceTransformer('all-miniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2382.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-miniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
embeddings = model.encode(data['Docs'].tolist(), batch_size = 64, show_progress_bar = True)

Batches:  23%|██▎       | 892/3879 [29:52<1:40:02,  2.01s/it] 


KeyboardInterrupt: 

In [ ]:
np.save("embeddings.npy", embeddings)

Retrieval

In [32]:
def retrieve_query(query, k = 5):
    query_vector = model.encode([query], convert_to_numpy = True)
    distance, indices = index.search(query_vector, k)

    results = data.iloc[indices[0]]
    results = results.sort_values(by = 'Confidence Score', ascending = False)

    return results